# 2. Station features for clustering

Collapses roughly 267,000 daily records into one row per station (N = 434), describing volume, variability and the weekly usage pattern.

## 2.1 Load the processed data

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

raw = pd.read_csv("../data/processed/TfL_Data_Processed.csv")
raw.head()

,TravelDate,DayOfWeek,Station,EntryTapCount,ExitTapCount,DayNum,Month,Year,IsWeekend
0,2024-01-01,Monday,Abbey Road DLR,395,375,0,1,2024,False
1,2024-01-01,Monday,Abbey Wood,5898,5963,0,1,2024,False
2,2024-01-01,Monday,Acton Central,609,474,0,1,2024,False
3,2024-01-01,Monday,Acton Main Line,1717,1710,0,1,2024,False
4,2024-01-01,Monday,Acton Town,2928,3334,0,1,2024,False


## 2.2 Station volume

In [2]:
mean_traffic = (raw[["Station", "EntryTapCount"]]
      .groupby("Station")
      .agg("mean")
      .assign(LogMeanEntryCount=lambda d: np.log10(d["EntryTapCount"]))
      )[["LogMeanEntryCount"]]

print(mean_traffic.head(), "\n")
mean_traffic.describe()

                 LogMeanEntryCount
Station                           
Abbey Road DLR            2.918674
Abbey Wood                4.189285
Acton Central             3.319563
Acton Main Line           3.591564
Acton Town                3.830877 



,LogMeanEntryCount
count,434.000000
mean,3.692834
std,0.455810
min,2.293106
25%,3.388516
50%,3.676271
75%,3.941855
max,5.020819


## 2.3 Variability

In [3]:
traffic_variability = (raw[["Station", "EntryTapCount"]]
      .groupby("Station")
      .agg("std")
      .assign(CoefVariation=lambda d: d["EntryTapCount"] / raw.groupby("Station")["EntryTapCount"].mean())
      .drop("EntryTapCount", axis="columns")
      )

print(traffic_variability.head(), "\n")
traffic_variability.describe()

                 CoefVariation
Station                       
Abbey Road DLR        0.308103
Abbey Wood            0.235205
Acton Central         0.298692
Acton Main Line       0.273722
Acton Town            0.256122 



,CoefVariation
count,434.000000
mean,0.270614
std,0.074068
min,0.139406
25%,0.220914
50%,0.256025
75%,0.303907
max,0.724067


## 2.4 Weekly usage pattern

In [4]:
day_rename_map = {0: "Monday",
                  1: "Tuesday",
                  2: "Wednesday",
                  3: "Thursday",
                  4: "Friday",
                  5: "Saturday",
                  6: "Sunday"}

weekly_share = (raw[["Station", "DayNum", "EntryTapCount"]]
                .groupby(["Station", "DayNum"])
                .mean()
                .assign(WeeklyShare=lambda d: d["EntryTapCount"] / d.groupby("Station")["EntryTapCount"].sum())
                .pivot_table(index="Station", columns="DayNum", values="WeeklyShare")
                .rename(day_rename_map, axis="columns")
                )
weekly_share

DayNum,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
Station,,,,,,,
Abbey Road DLR,0.147414,0.164390,0.166385,0.169606,0.166696,0.104933,0.080576
Abbey Wood,0.142862,0.163450,0.164835,0.168025,0.153667,0.123783,0.083377
Acton Central,0.141705,0.161932,0.161901,0.165735,0.164848,0.132519,0.071359
Acton Main Line,0.139768,0.157570,0.161694,0.169703,0.155022,0.135985,0.080257
Acton Town,0.137806,0.157164,0.159362,0.161145,0.159265,0.128871,0.096385
...,...,...,...,...,...,...,...
Woodgrange Park,0.144819,0.156692,0.158617,0.158772,0.160759,0.135495,0.084847
Woodside Park,0.140011,0.164714,0.166314,0.169398,0.152395,0.123418,0.083750
Woolwich Arsenal DLR,0.147322,0.160277,0.156150,0.154543,0.157218,0.122012,0.102479


In [5]:
d = (weekly_share
     .reset_index()
     .melt(id_vars="Station", var_name="Weekday", value_name="value")
     )

is_weekend = d["Weekday"].isin(["Saturday", "Sunday"])
weekday_avg = d[~is_weekend].groupby("Station")["value"].mean()
weekend_avg = d[is_weekend].groupby("Station")["value"].mean()

weekly_balance = (weekday_avg / weekend_avg).rename("WeekdayWeekendBalance")
weekly_balance

Station
Abbey Road DLR          1.756231
Abbey Wood              1.530867
Acton Central           1.561957
Acton Main Line         1.449781
Acton Town              1.375753
                          ...   
Woodgrange Park         1.415356
Woodside Park           1.530805
Woolwich Arsenal DLR    1.381812
Woolwich Arsenal NR     1.324380
Woolwich EL             1.295532
Name: WeekdayWeekendBalance, Length: 434, dtype: float64

## 2.5 Join the features

In [6]:
station_features = (mean_traffic
                    .join(traffic_variability, how="inner")
                    .join(weekly_share, how="inner")
                    .join(weekly_balance, how="inner"))
station_features.head()

,LogMeanEntryCount,CoefVariation,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,WeekdayWeekendBalance
Station,,,,,,,,,,
Abbey Road DLR,2.918674,0.308103,0.147414,0.164390,0.166385,0.169606,0.166696,0.104933,0.080576,1.756231
Abbey Wood,4.189285,0.235205,0.142862,0.163450,0.164835,0.168025,0.153667,0.123783,0.083377,1.530867
Acton Central,3.319563,0.298692,0.141705,0.161932,0.161901,0.165735,0.164848,0.132519,0.071359,1.561957
Acton Main Line,3.591564,0.273722,0.139768,0.157570,0.161694,0.169703,0.155022,0.135985,0.080257,1.449781
Acton Town,3.830877,0.256122,0.137806,0.157164,0.159362,0.161145,0.159265,0.128871,0.096385,1.375753


In [7]:
station_features.describe()

,LogMeanEntryCount,CoefVariation,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,WeekdayWeekendBalance
count,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000
mean,3.692834,0.270614,0.138683,0.159976,0.162964,0.166169,0.154014,0.128478,0.089717,1.491777
std,0.455810,0.074068,0.008322,0.011881,0.010973,0.010697,0.007236,0.020991,0.018012,0.375760
min,2.293106,0.139406,0.100787,0.091196,0.107725,0.130541,0.118420,0.050546,0.033075,0.766252
25%,3.388516,0.220914,0.134261,0.153891,0.157279,0.160837,0.150254,0.117256,0.078259,1.286385
50%,3.676271,0.256025,0.139367,0.159925,0.163149,0.165630,0.155180,0.128819,0.088309,1.447932
75%,3.941855,0.303907,0.144007,0.165167,0.168366,0.170955,0.158682,0.139126,0.099383,1.620942
max,5.020819,0.724067,0.176476,0.207456,0.210475,0.219164,0.169450,0.213065,0.164057,3.969604


In [8]:
missing_stations = len(station_features) - raw["Station"].nunique()

if missing_stations == 0:
    print("No stations dropped.")
else:
    print(missing_stations, "stations dropped.")

No stations dropped.


In [9]:
station_features.corr()
# Tuesday, Wednesday, Thursday, Saturday and Sunday each correlate with
# WeekdayWeekendBalance at |r| > 0.8, so including every day would let the weekly
# pattern dominate the distance metric. Friday is nearly independent of the balance
# and is kept as a separate feature.

,LogMeanEntryCount,CoefVariation,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,WeekdayWeekendBalance
LogMeanEntryCount,1.000000,-0.320475,-0.309430,-0.239766,-0.228132,-0.168049,-0.256704,0.290643,0.304336,-0.229053
CoefVariation,-0.320475,1.000000,0.185597,0.476511,0.549279,0.552227,-0.112416,-0.474911,-0.464066,0.597359
Monday,-0.309430,0.185597,1.000000,0.573172,0.460429,0.404997,0.220295,-0.789123,-0.530032,0.633095
Tuesday,-0.239766,0.476511,0.573172,1.000000,0.950121,0.872285,-0.273775,-0.876280,-0.890170,0.897079
Wednesday,-0.228132,0.549279,0.460429,0.950121,1.000000,0.901566,-0.266005,-0.843870,-0.893845,0.881754
Thursday,-0.168049,0.552227,0.404997,0.872285,0.901566,1.000000,-0.308001,-0.806621,-0.841918,0.868309
Friday,-0.256704,-0.112416,0.220295,-0.273775,-0.266005,-0.308001,1.000000,0.032064,-0.015320,-0.112536
Saturday,0.290643,-0.474911,-0.789123,-0.876280,-0.843870,-0.806621,0.032064,1.000000,0.757531,-0.895238
Sunday,0.304336,-0.464066,-0.530032,-0.890170,-0.893845,-0.841918,-0.015320,0.757531,1.000000,-0.848637
WeekdayWeekendBalance,-0.229053,0.597359,0.633095,0.897079,0.881754,0.868309,-0.112536,-0.895238,-0.848637,1.000000


In [10]:
station_features_selected = station_features[["LogMeanEntryCount", "CoefVariation", "Friday", "WeekdayWeekendBalance"]]
station_features_selected.head()

,LogMeanEntryCount,CoefVariation,Friday,WeekdayWeekendBalance
Station,,,,
Abbey Road DLR,2.918674,0.308103,0.166696,1.756231
Abbey Wood,4.189285,0.235205,0.153667,1.530867
Acton Central,3.319563,0.298692,0.164848,1.561957
Acton Main Line,3.591564,0.273722,0.155022,1.449781
Acton Town,3.830877,0.256122,0.159265,1.375753


## 2.6 Standardise

In [11]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
station_features_scaled = scaler.fit_transform(station_features_selected)
station_features_scaled.describe()

,LogMeanEntryCount,CoefVariation,Friday,WeekdayWeekendBalance
count,4.340000e+02,4.340000e+02,4.340000e+02,4.340000e+02
mean,8.063187e-16,-3.867874e-16,-2.730023e-15,-4.092988e-18
std,1.001154e+00,1.001154e+00,1.001154e+00,1.001154e+00
min,-3.074400e+00,-1.773500e+00,-4.924625e+00,-1.933048e+00
25%,-6.684109e-01,-6.717810e-01,-5.200891e-01,-5.472350e-01
50%,-3.637828e-02,-1.972009e-01,1.613618e-01,-1.168167e-01
75%,5.469561e-01,4.500086e-01,6.458891e-01,3.441406e-01
max,2.916822e+00,6.129144e+00,2.135788e+00,6.601787e+00


## 2.7 Save

In [12]:
os.makedirs("../models", exist_ok=True)

path = "../data/processed/TfL_Station_Features.csv"
station_features_scaled.to_csv(path, index=True)
print("Output saved to:", path)

scaler_path = "../models/scaler.joblib"
joblib.dump(scaler, scaler_path)
print("Scaler saved to:", scaler_path)

Output saved to: ../data/processed/TfL_Station_Features.csv
Scaler saved to: ../models/scaler.joblib
